In [14]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.pipeline import Pipeline


In [15]:

# Load the data
data_dir = '../data2'
data = pd.read_csv(data_dir + '/data.csv')
data.set_index('id', inplace=True)


In [16]:

# Selected features for recommendation
features = [
    'acousticness', 'danceability', 'duration_ms', 'energy', 'explicit', 
    'instrumentalness', 'key', 'liveness', 'loudness', 'mode', 
    'popularity', 'speechiness', 'tempo', 'valence', 'year'
]

# Preparing the data
sub_features = features.copy()
sub_features.remove('year')


In [17]:

pipeline = Pipeline([
    ('scaler', MinMaxScaler()),
    ('pca', PCA(n_components=0.95))  # Retaining 95% variance
])

# Fit the pipeline to the data and transform it
X_pca = pipeline.fit_transform(data[sub_features].values)


In [18]:
X_pca = pd.DataFrame(X_pca, columns=[f'PC{i+1}' for i in range(X_pca.shape[1])])
X_pca['year'] = data['year'].values


In [19]:
neigh = NearestNeighbors(n_neighbors=4)
neigh.fit(X_pca)


NearestNeighbors(n_neighbors=4)

In [20]:

# Preprocess the artist names
artists = data['artists'].str.split(',', expand=True)
data['artists_name'] = artists[0].replace({'\[': '', '\]': '', "'": ''}, regex=True)


In [28]:


# Function to recommend songs by artist similarity
def recommend_songs_by_id(song_id, num_recommendations=5):
    # Get the song's features
    song = data.loc[song_id, sub_features].values.reshape(1, -1)
    # Transform the features using the pipeline
    song_transformed = pipeline.transform(song)
    song_transformed = pd.DataFrame(song_transformed, columns=[f'PC{i+1}' for i in range(song_transformed.shape[1])])
    song_transformed['year'] = data.loc[song_id, 'year']
    
    # Find the nearest neighbors
    distances, indices = neigh.kneighbors(song_transformed, n_neighbors=num_recommendations+1)
    recommended_songs = data.iloc[indices[0]].index
    return recommended_songs[1:num_recommendations + 1]

# Function to recommend songs by artist similarity
def recommend_songs_by_artist(song_id, num_recommendations=5):
    artist_name = data.loc[song_id, 'artists_name']
    songs_by_artist = data[(data['artists_name'] == artist_name) & (data.index != song_id)].index
    
    # Get the song's features
    cos_sim = []
    for song in songs_by_artist:
        song_features = data.loc[song, sub_features].values.reshape(1, -1)
        song_features_transformed = pipeline.transform(song_features)
        cos_sim.append(cosine_similarity(song_features_transformed, pipeline.transform(data.loc[song_id, sub_features].values.reshape(1, -1))))
        
    cos_sim = np.array(cos_sim).reshape(-1)
    indices = np.argsort(cos_sim)[::-1]
    recommended_songs = songs_by_artist[indices]
    return recommended_songs[:num_recommendations]

# Combine recommendations
def recommend_songs(song_id, num_recommendations_by_id=3, num_recommendations_by_artist=2):
    rec_by_id = recommend_songs_by_id(song_id, num_recommendations_by_id)
    rec_by_artist = recommend_songs_by_artist(song_id, num_recommendations_by_artist)
    combined_recs = np.concatenate((rec_by_id, rec_by_artist))
    return data.loc[combined_recs[:num_recommendations_by_artist+num_recommendations_by_id], ['name', 'artists_name', 'year', 'popularity']]

# Example usage
song_id = '5SiZJoLXp3WOl3J4C8IK0d'
recommendations = recommend_songs(song_id, num_recommendations_by_id=3, num_recommendations_by_artist=2)
recommendations


,name,artists_name,year,popularity
id,,,,
1psvnQxSDdIKTDM2Jm8QKt,By Yourself (feat. Jhené Aiko & Mustard),Ty Dolla $ign,2020,69
4dPVmeisPfQrLcjx0Wz1KW,JoJo Pose,Apollo Fresh,2020,71
6EudabsFkMkxL9ZDVDqn0W,Kid That Kidd (feat. Future and Doe Boy),Trippie Redd,2020,69
5Z01UMMf7V1o0MzF86s6WJ,"Lose Yourself - From ""8 Mile"" Soundtrack",Eminem,2005,77
77Ft1RJngppZlq59B6uP0z,Lose Yourself,Eminem,2014,64


In [30]:
import joblib

# Save the pipeline
joblib.dump(pipeline, 'preprocessor.pkl')
# Save the model
joblib.dump(neigh, 'recommender.pkl')
# Save the data
data.to_csv('data.csv')
